In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv(r"C:\Users\ni\Desktop\Demand Forecaster\cleaned_data\cleaned_train.csv")

In [3]:
"""
Apply log transformation to target variable to stabilize variance
and improve model performance (especially RMSLE).
"""

train_df["log_sales"] = np.log1p(train_df["sales"])

In [4]:
"""
Define feature set and target variable for baseline model.
Exclude 'date' and target column from features.
"""

features = [
    "store_nbr",
    "family",
    "onpromotion",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7"
]

target ="log_sales"

In [5]:
"""
Encode categorical variables using Label Encoding.
"""

from sklearn.preprocessing import LabelEncoder

le_family = LabelEncoder()
train_df["family"] = le_family.fit_transform(train_df["family"])

In [6]:
train_df

,date,store_nbr,family,sales,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14,rolling_mean_7,log_sales
0,2013-01-22,1,0,1037.0,0,1,1,22,0,1104.0,1149.0,1029.0,1008.142857,6.945051
1,2013-01-23,1,0,1052.0,0,2,1,23,0,1037.0,1043.0,1186.0,992.142857,6.959399
2,2013-01-24,1,0,846.0,0,3,1,24,0,1052.0,898.0,847.0,993.428571,6.741701
3,2013-01-25,1,0,1103.0,0,4,1,25,0,846.0,1130.0,1033.0,986.000000,7.006695
4,2013-01-26,1,0,965.0,0,5,1,26,1,1103.0,1280.0,1143.0,982.142857,6.873164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1436827,2017-08-11,54,15,0.0,0,4,8,11,0,2.0,0.0,4.0,3.000000,0.000000
1436828,2017-08-12,54,15,1.0,1,5,8,12,1,0.0,3.0,4.0,3.000000,0.693147
1436829,2017-08-13,54,15,2.0,0,6,8,13,1,1.0,0.0,4.0,2.714286,1.098612
1436830,2017-08-14,54,15,0.0,0,0,8,14,0,2.0,0.0,4.0,3.000000,0.000000


In [7]:
"""
Split data based on time to simulate real forecasting.
Train on past data, validate on recent data.
"""

train_df["date"] = pd.to_datetime(train_df["date"])
split_date = train_df["date"].quantile(0.8)

train_data = train_df[train_df["date"] <= split_date]
val_data = train_df[train_df["date"] > split_date]

X_train = train_data[features]
y_train = train_data[target]

X_val = val_data[features]
y_val = val_data[target]

In [8]:
X_train

,store_nbr,family,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14,rolling_mean_7
0,1,0,0,1,1,22,0,1104.0,1149.0,1029.0,1008.142857
1,1,0,0,2,1,23,0,1037.0,1043.0,1186.0,992.142857
2,1,0,0,3,1,24,0,1052.0,898.0,847.0,993.428571
3,1,0,0,4,1,25,0,846.0,1130.0,1033.0,986.000000
4,1,0,0,5,1,26,1,1103.0,1280.0,1143.0,982.142857
...,...,...,...,...,...,...,...,...,...,...,...
1436495,54,15,0,0,9,12,0,0.0,3.0,0.0,1.000000
1436496,54,15,0,1,9,13,0,0.0,0.0,0.0,0.571429
1436497,54,15,0,2,9,14,0,0.0,0.0,2.0,0.571429
1436498,54,15,0,3,9,15,0,0.0,0.0,1.0,0.571429


In [9]:
X_val

,store_nbr,family,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14,rolling_mean_7
1331,1,0,54,5,9,17,1,2414.0,2251.0,2167.0,1954.571429
1332,1,0,40,6,9,18,1,2521.0,864.0,921.0,1993.142857
1333,1,0,48,0,9,19,0,1177.0,2453.0,2240.0,2037.857143
1334,1,0,54,1,9,20,0,2342.0,1942.0,2391.0,2022.000000
1335,1,0,57,2,9,21,0,2194.0,1992.0,2060.0,2058.000000
...,...,...,...,...,...,...,...,...,...,...,...
1436827,54,15,0,4,8,11,0,2.0,0.0,4.0,3.000000
1436828,54,15,1,5,8,12,1,0.0,3.0,4.0,3.000000
1436829,54,15,0,6,8,13,1,1.0,0.0,4.0,2.714286
1436830,54,15,0,0,8,14,0,2.0,0.0,4.0,3.000000


In [81]:
"""
Train a baseline Random Forest model for demand forecasting.
"""

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [82]:
y_pred_log = model.predict(X_val)

In [83]:
y_pred_log

array([7.69912893, 7.5434933 , 7.61411809, ..., 1.25137374, 1.17067017,
       0.95388711], shape=(286848,))

In [84]:
y_pred = np.expm1(y_pred_log)
y_val_actual = np.expm1(y_val)

In [85]:
"""
Evaluate model after reversing log transformation.
"""

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

# Ensure non-negative predictions
y_pred = np.maximum(0, y_pred)

# MAE
mae = mean_absolute_error(y_val_actual, y_pred)

# RMSE
rmse = np.sqrt(mean_squared_error(y_val_actual, y_pred))

# RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_val_actual, y_pred))

print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"RMSLE : {rmsle:.4f}")

MAE   : 150.6860
RMSE  : 499.2007
RMSLE : 0.3749


In [10]:
"""
Train LightGBM model optimized for MAE (L1 loss),
which is more suitable for inventory prediction.
"""

import lightgbm as lgb

model_new = lgb.LGBMRegressor(
    objective="regression_l1",   # MAE optimization
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=10,
    num_leaves=50,
    random_state=42,
    n_jobs=-1
)

model_new.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038981 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1384
[LightGBM] [Info] Number of data points in the train set: 1149984, number of used features: 11
[LightGBM] [Info] Start training from score 5.231109


,boosting_type,'gbdt'
,num_leaves,50
,max_depth,10
,learning_rate,0.03
,n_estimators,1500
,subsample_for_bin,200000
,objective,'regression_l1'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [11]:
y_pred_log = model_new.predict(X_val)

# Convert BOTH to original scale
y_pred = np.expm1(y_pred_log)
y_val_actual = np.expm1(y_val)

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_val_actual, y_pred)
print(f"MAE: {mae:.4f}")

MAE: 124.3172


In [43]:
train_df["sales"].mean()

np.float64(730.9225986081779)

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 2.6 MB/s eta 0:00:40
    --------------------------------------- 1.3/101.7 MB 2.7 MB/s eta 0:00:38
    --------------------------------------- 2.4/101.7 MB 3.1 MB/s eta 0:00:32
   - -------------------------------------- 3.1/101.7 MB 3.4 MB/s eta 0:00:30
   - -------------------------------------- 3.9/101.7 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 5.0/101.7 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 5.5/101.7 MB 3.7 MB/s eta 0:00:26
   -- ------------------------------------- 6.6/101.7 MB 3.7 MB/s eta 0:00:26
   -- ------------------------------------- 7.6/101.7 MB 3.8 MB/s eta 0:00:25
   --- ------------------------------------ 8.4/101.7 MB 3.8 MB/s eta 0:00:25
   --- 

  You can safely remove it manually.


In [90]:
"""
XGBoost baseline (MAE in actual units - FIXED)
"""

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# -------------------------------
# 1. FORCE correct target (units)
# -------------------------------
y_train_actual = train_data["sales"].copy()
y_val_actual = val_data["sales"].copy()

# -------------------------------
# 2. Define model
# -------------------------------
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# -------------------------------
# 3. Train
# -------------------------------
xgb_model.fit(X_train, y_train_actual)

# -------------------------------
# 4. Predict
# -------------------------------
y_pred_units = xgb_model.predict(X_val)
y_pred_units = np.maximum(0, y_pred_units)

# -------------------------------
# 5. Evaluate
# -------------------------------
mae = mean_absolute_error(y_val_actual, y_pred_units)

print(f"MAE (units): {mae:.2f}")

MAE (units): 142.56


In [13]:
import joblib

joblib.dump(model_new, r"C:\Users\ni\Desktop\credit_risk_predictor\final_model\demand_forecast_model.pkl")

['C:\\Users\\ni\\Desktop\\credit_risk_predictor\\final_model\\demand_forecast_model.pkl']

In [14]:
print(X_train.columns)

Index(['store_nbr', 'family', 'onpromotion', 'day_of_week', 'month',
       'day_of_month', 'is_weekend', 'lag_1', 'lag_7', 'lag_14',
       'rolling_mean_7'],
      dtype='object')


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
import joblib

# Columns
categorical_cols = ["family"]
numeric_cols = [
    "store_nbr",
    "onpromotion",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7"
]

# IMPORTANT: Use OrdinalEncoder (same idea as LabelEncoder)
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ("num", "passthrough", numeric_cols)
])

# Full pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model_new)
])

# Train
pipeline.fit(X_train, y_train)

# Save
joblib.dump(pipeline, r"C:\Users\ni\Desktop\credit_risk_predictor\final_model\demand_pipeline.pkl")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057040 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1384
[LightGBM] [Info] Number of data points in the train set: 1149984, number of used features: 11
[LightGBM] [Info] Start training from score 5.231109


['C:\\Users\\ni\\Desktop\\credit_risk_predictor\\final_model\\demand_pipeline.pkl']

In [18]:
model = joblib.load( r"C:\Users\ni\Desktop\credit_risk_predictor\final_model\demand_pipeline.pkl")
print(model.predict(X_val[:5]))

[7.80530738 7.039324   7.71430812 7.72610755 7.74191102]


C:\Users\ni\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
